# S3: Cross-Domain Transfer (PlantVillage → PlantDoc)

**Leaf-JEPA IRP** | Stage 5 — PEFT Adaptation Experiments

**Research Question:** RQ4 — Does Leaf-JEPA improve cross-domain transfer?

**No training on PlantDoc.** All models were trained on PlantVillage only.
Evaluation uses k-NN on frozen features + trained head zero-shot transfer.

**Key comparison:** Generic I-JEPA vs Leaf-JEPA on PlantDoc (same PEFT method, different encoder).

## Imports & Configuration

In [3]:
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

from stage5_peft_adaptation_experiments.config_stage5 import *
from stage5_peft_adaptation_experiments.peft_utils import (
    set_seed, get_device, load_ijepa_encoder, extract_features,
    knn_evaluate, PlantVillageDataset, PlantDocDataset,
    load_split, save_results, load_results, plot_confusion_matrix,
)

from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

verify_config()
device = get_device()
set_seed(RANDOM_SEED)

SET3_DIR = PEFT_DIR / "set3"
SET3_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / "set3_confusion_matrices").mkdir(parents=True, exist_ok=True)

Stage 5 Configuration Verification
  ✅  PlantVillage root exists
  ✅  PlantDoc root exists
  ✅  Splits directory exists
  ✅  Normalisation stats exist
  ✅  NORM_MEAN matches JSON
  ✅  NORM_STD matches JSON
  ✅  I-JEPA checkpoint exists
  ✅  Leaf-JEPA checkpoint exists
  ✅  Baselines directory exists
  ✅  WANDB_ENTITY set

  10/10 checks passed
  🚀  All checks passed — ready to run Stage 5
Using GPU: NVIDIA H200
VRAM: 150.0 GB


## Load Taxonomy & Filter PlantDoc Classes

In [6]:
# Only evaluate on overlapping classes with >= PD_MIN_SAMPLES samples

taxonomy = {}
if TAXONOMY_PATH.exists():
    taxonomy = json.loads(TAXONOMY_PATH.read_text())
    print(f"✅ Loaded taxonomy from {TAXONOMY_PATH.name}")

# Build PV <-> PD class mapping
pv_to_pd_map = {}
pd_valid_classes = []

if taxonomy:
    for entry in taxonomy.get("entries", []): 
        pv_name = entry.get("plantvillage_label", "")
        pd_name = entry.get("plantdoc_label", "")
        
        if pv_name and pd_name: # This will be False if pd_name is None/null
            pv_to_pd_map[pv_name] = pd_name
            pd_valid_classes.append(pd_name)
            

# Fallback: directory name matching
if not pd_valid_classes:
    print("⚠️  Taxonomy JSON not loaded — using directory name matching")
    pv_classes = set(d.name for d in PV_ROOT.iterdir() if d.is_dir())
    pd_classes = set(d.name for d in PD_ROOT.iterdir() if d.is_dir())
    pd_valid_classes = list(pv_classes & pd_classes)

# Filter by minimum sample count
pd_class_counts = {}
for cls_dir in PD_ROOT.iterdir():
    if cls_dir.is_dir() and cls_dir.name in pd_valid_classes:
        n_imgs = len([
            f for f in cls_dir.iterdir()
            if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        ])
        pd_class_counts[cls_dir.name] = n_imgs

pd_eval_classes = [c for c, n in pd_class_counts.items() if n >= PD_MIN_SAMPLES]

print(f"\nPlantDoc evaluation classes (>= {PD_MIN_SAMPLES} samples): {len(pd_eval_classes)}")
for c in sorted(pd_eval_classes):
    print(f"  {c}: {pd_class_counts[c]} samples")

✅ Loaded taxonomy from taxonomy.json

PlantDoc evaluation classes (>= 10 samples): 17
  Apple Scab Leaf: 93 samples
  Apple leaf: 91 samples
  Corn Gray leaf spot: 68 samples
  Corn leaf blight: 192 samples
  Corn rust leaf: 116 samples
  Peach leaf: 112 samples
  Potato leaf early blight: 117 samples
  Potato leaf late blight: 105 samples
  Tomato Early blight leaf: 88 samples
  Tomato Septoria leaf spot: 151 samples
  Tomato leaf: 63 samples
  Tomato leaf bacterial spot: 110 samples
  Tomato leaf late blight: 111 samples
  Tomato leaf mosaic virus: 54 samples
  Tomato leaf yellow virus: 76 samples
  grape leaf: 69 samples
  grape leaf black rot: 64 samples


## Build Dataloaders

In [8]:
eval_tf = get_eval_transform()

# PlantVillage training features (for kNN reference)
pv_train_ds = PlantVillageDataset(PV_ROOT, load_split(SPLITS_DIR, "plantvillage_train"), eval_tf)
pv_train_loader = DataLoader(
    pv_train_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# PlantDoc evaluation
pd_ds = PlantDocDataset(PD_ROOT, transform=eval_tf, class_filter=pd_eval_classes)
pd_loader = DataLoader(
    pd_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"PV train: {len(pv_train_ds):,} images")
print(f"PD eval:  {len(pd_ds):,} images, {len(pd_ds.class_names)} classes")

PV train: 38,013 images
PD eval:  1,680 images, 17 classes


## Extract Features — Generic I-JEPA

In [9]:
print("Extracting features: Generic I-JEPA encoder")
print("-" * 50)

generic_encoder = load_ijepa_encoder(
    IJEPA_CHECKPOINT, VIT_MODEL_NAME, VIT_EMBED_DIM, device
)

generic_pv_feats, generic_pv_labels = extract_features(
    generic_encoder, pv_train_loader, device
)
generic_pd_feats, generic_pd_labels = extract_features(
    generic_encoder, pd_loader, device
)

print(f"  PV features: {generic_pv_feats.shape}")
print(f"  PD features: {generic_pd_feats.shape}")

del generic_encoder
torch.cuda.empty_cache()

Extracting features: Generic I-JEPA encoder
--------------------------------------------------
  Encoder loaded from IN1K-vit.h.14-300e.pth.tar (frozen)
  PV features: (38013, 257, 1280)
  PD features: (1680, 257, 1280)


## Extract Features — Leaf-JEPA

In [ ]:
print("Extracting features: Leaf-JEPA encoder")
print("-" * 50)

leaf_encoder = load_ijepa_encoder(
    LEAF_JEPA_CHECKPOINT, VIT_MODEL_NAME, VIT_EMBED_DIM, device
)

leaf_pv_feats, leaf_pv_labels = extract_features(
    leaf_encoder, pv_train_loader, device
)
leaf_pd_feats, leaf_pd_labels = extract_features(
    leaf_encoder, pd_loader, device
)

print(f"  PV features: {leaf_pv_feats.shape}")
print(f"  PD features: {leaf_pd_feats.shape}")

del leaf_encoder
torch.cuda.empty_cache()

Extracting features: Leaf-JEPA encoder
--------------------------------------------------
  Encoder loaded from leafjepa-vit-h14-best.pth (frozen)


## k-NN Cross-Domain Evaluation

In [ ]:
cross_domain_results = {}

print("\n" + "=" * 60)
print("  k-NN Cross-Domain Evaluation (PlantDoc)")
print("=" * 60)

print("\nGeneric I-JEPA:")
cross_domain_results["generic_ijepa_knn"] = knn_evaluate(
    generic_pv_feats, generic_pv_labels,
    generic_pd_feats, generic_pd_labels,
    k_values=KNN_K_VALUES,
)

print("\nLeaf-JEPA:")
cross_domain_results["leaf_jepa_knn"] = knn_evaluate(
    leaf_pv_feats, leaf_pv_labels,
    leaf_pd_feats, leaf_pd_labels,
    k_values=KNN_K_VALUES,
)

# RQ4 answer
g_f1 = cross_domain_results["generic_ijepa_knn"]["best_macro_f1"]
l_f1 = cross_domain_results["leaf_jepa_knn"]["best_macro_f1"]
delta = l_f1 - g_f1

print(f"\n{'='*60}")
print(f"  ★ RQ4 DELTA (Leaf-JEPA − Generic I-JEPA on PlantDoc)")
print(f"{'='*60}")
print(f"  Generic I-JEPA: {g_f1:.4f}")
print(f"  Leaf-JEPA:      {l_f1:.4f}")
print(f"  Delta:          {delta:+.4f}")

if delta > 0:
    print(f"  → Domain pretraining IMPROVED cross-domain transfer")
else:
    print(f"  → Domain pretraining did NOT improve cross-domain transfer")

## t-SNE Domain Gap Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

comparisons = [
    ("Generic I-JEPA", generic_pv_feats, generic_pd_feats),
    ("Leaf-JEPA", leaf_pv_feats, leaf_pd_feats),
]

for ax, (name, pv_feats, pd_feats) in zip(axes, comparisons):
    # Subsample for speed
    n_pv = min(2000, len(pv_feats))
    n_pd = min(len(pd_feats), 1000)
    
    rng = np.random.RandomState(42)
    idx_pv = rng.choice(len(pv_feats), n_pv, replace=False)
    idx_pd = rng.choice(len(pd_feats), n_pd, replace=False)
    
    combined = np.concatenate([pv_feats[idx_pv], pd_feats[idx_pd]])
    labels = ["PlantVillage"] * n_pv + ["PlantDoc"] * n_pd
    
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    embedded = tsne.fit_transform(combined)
    
    for dataset, colour, marker in [
        ("PlantVillage", "#3498db", "o"),
        ("PlantDoc", "#e74c3c", "^"),
    ]:
        mask = np.array(labels) == dataset
        ax.scatter(
            embedded[mask, 0], embedded[mask, 1],
            c=colour, marker=marker, s=15, alpha=0.5, label=dataset,
        )
    
    ax.set_title(name, fontsize=14)
    ax.legend(fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    "Domain Gap: PlantVillage vs PlantDoc Feature Distributions",
    fontsize=16,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "set3_domain_gap_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {FIGURES_DIR / 'set3_domain_gap_tsne.png'}")

## Summary Bar Chart & Save

In [ ]:
# Bar chart
methods = list(cross_domain_results.keys())
f1_scores = [cross_domain_results[m]["best_macro_f1"] for m in methods]

fig, ax = plt.subplots(figsize=(10, 5))
colours = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6"][:len(methods)]
bars = ax.bar(range(len(methods)), f1_scores, color=colours)

ax.set_xticks(range(len(methods)))
ax.set_xticklabels([m.replace("_", "\n") for m in methods], fontsize=10)
ax.set_ylabel("Macro-F1 on PlantDoc", fontsize=12)
ax.set_title("Cross-Domain Transfer: PlantDoc Evaluation", fontsize=14)

for bar, score in zip(bars, f1_scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f"{score:.3f}",
        ha="center", fontsize=11,
    )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "set3_cross_domain_bar.png", dpi=150)
plt.show()

save_results(cross_domain_results, PEFT_DIR / "set3_cross_domain.json")
print("\n✅ Set 3 complete.")